# Preprocess of IMDB data to create an IMDB content dataset

## Imports

In [11]:
import numpy as np
import pandas as pd
import zipfile as zipfile

import sys
import requests
import os

In [12]:
# URL of datasources
url_mdl                = 'https://files.grouplens.org/datasets/movielens/ml-20m.zip'

url_imdb_namebasics    = 'https://datasets.imdbws.com/name.basics.tsv.gz'
url_imdb_titleakas     = 'https://datasets.imdbws.com/title.akas.tsv.gz'
url_imdb_titlecrew     = 'https://datasets.imdbws.com/title.crew.tsv.gz'
url_imdb_titlebasics   = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_imdb_titleepisode  = 'https://datasets.imdbws.com/title.episode.tsv.gz'
url_imdb_titleprincipals  = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_imdb_titleratings  = 'https://datasets.imdbws.com/title.ratings.tsv.gz'

## Functions

## Preprocess

### limitation of dataset

In [13]:
# Common films between MDL and IMDB datasets
imdb_mdl_mapp = pd.read_csv('data_processed/imdb_mdl_mapp.csv', dtype={'tconst':object, 'movieId':object, 'primaryTitle':object, 'startYear':int})

# Reference list to limit datasets
list_movies_imdb = imdb_mdl_mapp[(imdb_mdl_mapp['startYear']>2000) & (imdb_mdl_mapp['startYear']<2010)]['tconst'].tolist()

#print(imdb_mdl_mapp.info())
#imdb_mdl_mapp.head(3)

### Basic info

In [14]:
# Load
dict_types = {'tconst':object, 'titleType':object, 'primaryTitle':object, 'startYear':int, 'runtimeMinutes':int, 'genres':object}
title_basics = pd.read_csv('data_imdb/title.basics.tsv.gz',compression='gzip', sep='\t', na_values=['\\N', 'nan', 'NA']) #,  dtype=dict_types

# preprocess of title_crew.tconst to match imdbId from mdl
    #title_basics['tconst_short'] = [x[-7:] for x in title_basics['tconst']]
    #title_basics['tconst_short'] = title_basics['tconst'].astype('int64')

# Limitation of the data set
title_basics = title_basics[title_basics['tconst'].isin(list_movies_imdb)]


# Drop of rows containing errors
i = title_basics[ title_basics['runtimeMinutes'] == 'Reality-TV' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)

i = title_basics[ title_basics['runtimeMinutes'] == 'Talk-Show' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)


i = title_basics[ title_basics['runtimeMinutes'] == 'Documentary' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)


i = title_basics[ title_basics['runtimeMinutes'] == 'Game-Show' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)


i = title_basics[ title_basics['runtimeMinutes'] == 'Animation,Comedy,Family' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)


i = title_basics[ title_basics['runtimeMinutes'] == 'Game-Show,Reality-TV' ].index.tolist()
title_basics = title_basics.drop(i, axis=0)


# NANs handling
title_basics['runtimeMinutes'] = title_basics['runtimeMinutes'].fillna(0)

    # title_basics['runtimeMinutes'] = title_basics['runtimeMinutes'].fillna(title_basics['runtimeMinutes'].mod()[0])

    # title_basics[title_basics['titleType'] == 'movie']['runtimeMinutes']        = title_basics[title_basics['titleType'] == 'movie']['runtimeMinutes'].fillna(60)
    # title_basics[title_basics['titleType'] == 'tvMovie']['runtimeMinutes']      = title_basics[title_basics['titleType'] == 'tvMovie']['runtimeMinutes'].fillna(50)
    # title_basics[title_basics['titleType'] == 'video']['runtimeMinutes']        = title_basics[title_basics['titleType'] == 'video']['runtimeMinutes'].fillna(20)
    # title_basics[title_basics['titleType'] == 'tvMiniSeries']['runtimeMinutes'] = title_basics[title_basics['titleType'] == 'tvMiniSeries']['runtimeMinutes'].fillna(20)
    # title_basics[title_basics['titleType'] == 'tvSpecial']['runtimeMinutes']    = title_basics[title_basics['titleType'] == 'tvSpecial']['runtimeMinutes'].fillna(30)
    # title_basics[title_basics['titleType'] == 'tvEpisode']['runtimeMinutes']    = title_basics[title_basics['titleType'] == 'tvEpisode']['runtimeMinutes'].fillna(45)
    # title_basics[title_basics['titleType'] == 'tvSeries']['runtimeMinutes']     = title_basics[title_basics['titleType'] == 'tvSeries']['runtimeMinutes'].fillna(45)
    # title_basics[title_basics['titleType'] == 'videoGame']['runtimeMinutes']    = title_basics[title_basics['titleType'] == 'videoGame']['runtimeMinutes'].fillna(15)
    # title_basics[title_basics['titleType'] == 'short']['runtimeMinutes']        = title_basics[title_basics['titleType'] == 'short']['runtimeMinutes'].fillna(5)
    # title_basics[title_basics['titleType'] == 'tvShort']['runtimeMinutes']      = title_basics[title_basics['titleType'] == 'tvShort']['runtimeMinutes'].fillna(5)


# Discretisation of runtime
title_basics['runtimeMinutes'] = title_basics['runtimeMinutes'].astype(float)

title_basics['runtimeCategory'] = pd.cut(x = title_basics['runtimeMinutes'],
                                         bins = [0.0, 10.0, 20.0, 30.0, 45.0, 60.0, 120.0, 150.0, 180.0, 9999.0],
                                         labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I'],
                                         include_lowest=True)


# Discretisation of year
title_basics['startYear'] = title_basics['startYear'].astype(float)

title_basics['yearCategory'] = pd.cut(x = title_basics['startYear'],
                                         bins = [1850.0, 1900.0, 1930.0, 1950.0, 1960.0, 1970.0, 1980.0, 1990.0, 2000.0, 2010.0, 2020.0, 2030.0],
                                         labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K'],
                                         include_lowest=True)



# Columns clean-up
title_basics = title_basics.drop(['originalTitle', 'isAdult', 'endYear'], axis=1)

print(title_basics['runtimeCategory'].max())
print(title_basics.info())
title_basics.head(4)

/tmp/ipykernel_28288/2929418184.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv('data_imdb/title.basics.tsv.gz',compression='gzip', sep='\t', na_values=['\\N', 'nan', 'NA']) #,  dtype=dict_types


I
<class 'pandas.core.frame.DataFrame'>
Int64Index: 8501 entries, 34803 to 7023346
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   tconst           8501 non-null   object  
 1   titleType        8501 non-null   object  
 2   primaryTitle     8501 non-null   object  
 3   startYear        8501 non-null   float64 
 4   runtimeMinutes   8501 non-null   float64 
 5   genres           8487 non-null   object  
 6   runtimeCategory  8501 non-null   category
 7   yearCategory     8501 non-null   category
dtypes: category(2), float64(2), object(4)
memory usage: 482.2+ KB
None


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,runtimeCategory,yearCategory
34803,tt0035423,movie,Kate & Leopold,2001.0,118.0,"Comedy,Fantasy,Romance",F,I
93938,tt0096056,movie,Crime and Punishment,2002.0,126.0,Drama,G,I
115402,tt0118141,movie,What Is It?,2005.0,72.0,Drama,F,I
115828,tt0118589,movie,Glitter,2001.0,104.0,"Drama,Music,Romance",F,I


### Add of director

In [15]:
# Load : A ajouter : chargement direct depuis url
title_crew = pd.read_csv('data_imdb/title.crew.tsv.gz',compression='gzip', sep='\t', na_values=['\\N', 'nan']) # note : \\N is required to avoid confusion with newline '\n' 

# Limitation of the data set
title_crew = title_crew[title_crew['tconst'].isin(list_movies_imdb)]

# Gestions de NANs
title_crew = title_crew.fillna('')
#title_crew = title_crew.dropna(axis=0, how='any', subset=['directors'])

title_crew.head(5)
#title_crew.isna().sum()

,tconst,directors,writers
34803,tt0035423,nm0003506,"nm0737216,nm0003506"
93938,tt0096056,nm0324875,"nm0234502,nm0324875"
115402,tt0118141,nm0000417,nm0000417
115828,tt0118589,nm0193554,"nm0921985,nm0486824"
116152,tt0118926,nm0000518,nm0787649


In [16]:
# Load Name
name_basics = pd.read_csv('data_imdb/name.basics.tsv.gz',compression='gzip', sep='\t', usecols=['nconst', 'primaryName'])

print(name_basics.info())
name_basics.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12354442 entries, 0 to 12354441
Data columns (total 2 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   nconst       object
 1   primaryName  object
dtypes: object(2)
memory usage: 188.5+ MB
None


,nconst,primaryName
0,nm0000001,Fred Astaire
1,nm0000002,Lauren Bacall
2,nm0000003,Brigitte Bardot
3,nm0000004,John Belushi
4,nm0000005,Ingmar Bergman


In [17]:
# Merge to add names
title_crew = title_crew.merge(right=name_basics, left_on='directors', right_on='nconst', how='left')
title_crew.head(5)

,tconst,directors,writers,nconst,primaryName
0,tt0035423,nm0003506,"nm0737216,nm0003506",nm0003506,James Mangold
1,tt0096056,nm0324875,"nm0234502,nm0324875",nm0324875,Menahem Golan
2,tt0118141,nm0000417,nm0000417,nm0000417,Crispin Glover
3,tt0118589,nm0193554,"nm0921985,nm0486824",nm0193554,Vondie Curtis-Hall
4,tt0118926,nm0000518,nm0787649,nm0000518,John Malkovich


In [18]:
# Aggregation of directors in list
title_crew = title_crew.groupby('tconst')['primaryName'].apply(list)
title_crew = pd.DataFrame(title_crew).reset_index()

# Transform list => string
title_crew['directorName'] = [' '.join(map(str, l)) for l in title_crew['primaryName']]

# Clean of dataframe
title_crew = title_crew.drop(['primaryName'], axis=1)

print(title_crew.info())
title_crew.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8501 entries, 0 to 8500
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tconst        8501 non-null   object
 1   directorName  8501 non-null   object
dtypes: object(2)
memory usage: 133.0+ KB
None


,tconst,directorName
0,tt0035423,James Mangold
1,tt0096056,Menahem Golan
2,tt0118141,Crispin Glover
3,tt0118589,Vondie Curtis-Hall
4,tt0118926,John Malkovich


## Merge

In [19]:
imdb_content =  title_basics.merge(right=title_crew, left_on='tconst', right_on='tconst', how='left')

#imdb_content = imdb_content.drop(['primaryName'], axis=1)

print(imdb_content.info())
imdb_content.head(5)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8501 entries, 0 to 8500
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   tconst           8501 non-null   object  
 1   titleType        8501 non-null   object  
 2   primaryTitle     8501 non-null   object  
 3   startYear        8501 non-null   float64 
 4   runtimeMinutes   8501 non-null   float64 
 5   genres           8487 non-null   object  
 6   runtimeCategory  8501 non-null   category
 7   yearCategory     8501 non-null   category
 8   directorName     8501 non-null   object  
dtypes: category(2), float64(2), object(5)
memory usage: 548.7+ KB
None


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,runtimeCategory,yearCategory,directorName
0,tt0035423,movie,Kate & Leopold,2001.0,118.0,"Comedy,Fantasy,Romance",F,I,James Mangold
1,tt0096056,movie,Crime and Punishment,2002.0,126.0,Drama,G,I,Menahem Golan
2,tt0118141,movie,What Is It?,2005.0,72.0,Drama,F,I,Crispin Glover
3,tt0118589,movie,Glitter,2001.0,104.0,"Drama,Music,Romance",F,I,Vondie Curtis-Hall
4,tt0118926,movie,The Dancer Upstairs,2002.0,132.0,"Crime,Drama,Thriller",G,I,John Malkovich


## Save file

In [20]:
imdb_content.to_csv("data_processed/imdb_content.csv.zip", index=False, compression="zip")